<a href="https://colab.research.google.com/github/racoope70/daytrading-with-ml/blob/main/kmeans_live_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!Protocol Buffer Fix (for TensorFlow)
!pip install --upgrade protobuf
!pip install protobuf==3.20.3

In [2]:
!pip install tensorflow

In [3]:
!pip install stable-baselines3[extra] gymnasium gym-anytrading yfinance xgboost joblib

In [4]:
import torch
import cudf
import cuml
import dask
import pandas as pd
import numpy as np
import scipy
import lightgbm as lgb
import gymnasium as gym
import stable_baselines3

#Version Checks
print("Library Versions")
print("--------------------")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("cuDF:", cudf.__version__)
print("cuML:", cuml.__version__)
print("Dask:", dask.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("LightGBM:", lgb.__version__)
print("Gymnasium:", gym.__version__)
print("Stable Baselines3:", stable_baselines3.__version__)

#GPU Check (Torch + NVIDIA)
print("\nGPU Availability")
print("--------------------")
print("PyTorch GPU Available:", torch.cuda.is_available())
print("GPU Count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))


In [5]:
#Core Libraries
import gc
import json
import os
import pickle
import sys
import time
from collections import defaultdict, deque
from datetime import datetime

#Data Science Essentials
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numba
import IPython.display as display

#Machine Learning & Data Processing
import joblib
import lightgbm as lgb
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import MinMaxScaler

#Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential, load_model

#RAPIDS Libraries (cuDF & cuML for GPU acceleration)
import cupy as cp

#Reinforcement Learning (Stable Baselines3)
import stable_baselines3
from stable_baselines3 import A2C, DDPG, DQN, PPO, SAC, TD3
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.logger import configure
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

#Gym & Trading Environments
import gym
import gymnasium as gym
import gym_anytrading
from gym.spaces import Box
from gymnasium.spaces import Box as GymBox, Discrete
from gymnasium.wrappers import TimeLimit
from gym_anytrading.envs import StocksEnv

#Financial & Stock Data Libraries
import yfinance as yf

#PyTorch Essentials
import torch
import torch.nn as nn
import torch.optim as optim


In [6]:
#Set CUDA Paths (Ensuring GPU Utilization)
os.environ['CUDA_HOME'] = '/usr/local/cuda-11.8'
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-11.8/lib64'

In [7]:
!nvidia-smi

In [8]:
import yfinance as yf
import time
import pandas as pd
from datetime import datetime, timedelta

#Setup
RESULTS_DIR = "results/kmeans_walkforward"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/data", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/signals", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/plots", exist_ok=True)

#Ticker list
TICKERS = [
    'AAPL', 'TSLA', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'BRK-B', 'JPM', 'JNJ',
    'XOM', 'V', 'PG', 'UNH', 'MA', 'HD', 'LLY', 'MRK', 'PEP', 'KO',
    'BAC', 'ABBV', 'AVGO', 'PFE', 'COST', 'CSCO', 'TMO', 'ABT', 'ACN', 'WMT',
    'MCD', 'ADBE', 'DHR', 'CRM', 'NKE', 'INTC', 'QCOM', 'NEE', 'AMD', 'TXN',
    'AMGN', 'UPS', 'LIN', 'PM', 'UNP', 'BMY', 'LOW', 'RTX', 'CVX', 'IBM',
    'GE', 'SBUX', 'ORCL'
]
SEQUENCE_LENGTH = 60
EPOCHS = 10
BATCH_SIZE = 32
LSTM_DIR = "results/lstm_walkforward"
KMEANS_DIR = "results/kmeans_walkforward"
os.makedirs(LSTM_DIR, exist_ok=True)
os.makedirs(KMEANS_DIR, exist_ok=True)

RESULTS_DIR = "results/kmeans_walkforward"
os.makedirs(RESULTS_DIR, exist_ok=True)

ANALYZE_METHODS = True
anomaly_eval_summary = []

def download_data(ticker, retries=3, sleep_time=10):
    end_date = datetime.today()
    start_date = end_date - timedelta(days=720)
    start_str = start_date.strftime('%Y-%m-%d')
    end_str = end_date.strftime('%Y-%m-%d')

    for attempt in range(retries):
        try:
            df = yf.download(ticker, start=start_str, end=end_str, interval="1h", progress=False)
            if not df.empty:
                df.reset_index(inplace=True)
                df['Datetime'] = pd.to_datetime(df['Datetime'])
                return df
        except Exception as e:
            print(f"Error downloading {ticker} (Attempt {attempt + 1}): {e}")
            time.sleep(sleep_time)
    print(f"iled to download {ticker} after {retries} attempts.")
    return None




In [9]:
#Utility Functions

def fix_dataframe_index(df):
    """
    Flattens MultiIndex columns and removes duplicated columns.
    """
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df.loc[:, ~df.columns.duplicated()]


def calculate_rsi(series, period=14):
    """
    Calculates the Relative Strength Index (RSI) for a given price series.
    """
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / (loss + 1e-6)
    return 100 - (100 / (1 + rs))


def compute_technical_indicators(df):
    """
    Applies technical indicators including RSI, MACD, Bollinger Bands, Stochastic Oscillator,
    OBV, CCI, momentum, trend, and volatility indicators.
    """
    df = df.copy()
    df = fix_dataframe_index(df)

    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / (loss + 1e-6)
    df['RSI'] = 100 - (100 / (1 + rs))

    ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema_12 - ema_26
    df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()

    df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()

    df['SMA_20'] = df['Close'].rolling(window=20).mean()
    df['STD_20'] = df['Close'].rolling(window=20).std()
    df['Upper_Band'] = df['SMA_20'] + 2 * df['STD_20']
    df['Lower_Band'] = df['SMA_20'] - 2 * df['STD_20']

    df['Lowest_Low'] = df['Low'].rolling(window=14).min()
    df['Highest_High'] = df['High'].rolling(window=14).max()
    denominator = (df['Highest_High'] - df['Lowest_Low']).replace(0, np.nan)
    df['Stoch'] = ((df['Close'] - df['Lowest_Low']) / denominator) * 100

    df['volatility'] = df['Close'].pct_change().rolling(20).std()
    df['ROC'] = df['Close'].pct_change(periods=10)

    typical_price = (df['High'] + df['Low'] + df['Close']) / 3
    df['CCI'] = (typical_price - typical_price.rolling(20).mean()) / (
        0.015 * typical_price.rolling(20).std()
    )

    df['PROC'] = ((df['Close'] - df['Close'].shift(12)) / df['Close'].shift(12)) * 100

    df['Rolling_Mean_50'] = df['Close'].rolling(window=50).mean()
    df['Expanding_Mean'] = df['Close'].expanding(min_periods=1).mean()

    df.dropna(inplace=True)
    return df


def generate_trade_labels(df, lookahead=10, threshold_factor=2):
    """
    Generates binary and dynamic trade labels based on future returns
    and volatility-adjusted thresholds.
    """
    df = df.copy()
    df = fix_dataframe_index(df)

    if 'Close' not in df.columns:
        raise KeyError("'Close' column is missing. Cannot generate trade labels.")

    df['Future_Close'] = df['Close'].shift(-lookahead)
    df['Price_Change'] = (df['Future_Close'] - df['Close']) / df['Close']
    df['Target'] = np.where(df['Price_Change'] > 0.03, 1, 0)

    buy_threshold = df['volatility'] * threshold_factor
    sell_threshold = -df['volatility'] * threshold_factor

    df['Dynamic_Label'] = np.where(
        df['Price_Change'] > buy_threshold, 1,
        np.where(df['Price_Change'] < sell_threshold, -1, 0)
    )

    df.dropna(inplace=True)
    return df


def drop_low_importance_features(df, feature_importance_df, threshold=1.0):
    """
    Drops low-importance features based on a given threshold.
    """
    low_importance_features = feature_importance_df[
        feature_importance_df['importance'] < threshold
    ]['feature'].tolist()

    if low_importance_features:
        df.drop(columns=low_importance_features, inplace=True)
        print(f"Dropped low-importance features: {low_importance_features}")
    else:
        print("No low-importance features found to drop.")

    return df


In [13]:
# === Imports ===
import os
import gc
import json
import joblib
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from google.colab import drive

# === Mount Google Drive ===
drive.mount('/content/drive')

# === Configuration ===
RESULTS_DIR = "/content/drive/MyDrive/Results_May_2025/kmeans_walkforward_results"
FINAL_MODEL_DIR = "/content/drive/MyDrive/Results_May_2025/final_combined_results/models"
os.makedirs(f"{RESULTS_DIR}/plots", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/data", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

TICKERS = [
    'AAPL', 'TSLA', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'BRK-B', 'JPM', 'JNJ',
    'XOM', 'V', 'PG', 'UNH', 'MA', 'HD', 'LLY', 'MRK', 'PEP', 'KO',
    'BAC', 'ABBV', 'AVGO', 'PFE', 'COST', 'CSCO', 'TMO', 'ABT', 'ACN', 'WMT',
    'MCD', 'ADBE', 'DHR', 'CRM', 'NKE', 'INTC', 'QCOM', 'NEE', 'AMD', 'TXN',
    'AMGN', 'UPS', 'LIN', 'PM', 'UNP', 'BMY', 'LOW', 'RTX', 'CVX', 'IBM',
    'GE', 'SBUX', 'ORCL'
]

# === Technical Indicators ===
def compute_technical_indicators(df):
    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['STD_20'] = df['Close'].rolling(20).std()
    df['Upper_Band'] = df['SMA_20'] + 2 * df['STD_20']
    df['Lower_Band'] = df['SMA_20'] - 2 * df['STD_20']
    df['Momentum'] = df['Close'].diff(4)
    df['Volatility'] = df['Close'].pct_change().rolling(10).std()
    df.dropna(inplace=True)
    return df

# === KMeans Walkforward Evaluation ===
def walkforward_kmeans(df, n_clusters=3):
    df = compute_technical_indicators(df)
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df = df.sort_values('Datetime')

    latest_date = df['Datetime'].max()
    train_start = latest_date - pd.Timedelta(days=730)
    train_end = latest_date - pd.Timedelta(days=365)
    test_end = latest_date

    train_df = df[(df['Datetime'] >= train_start) & (df['Datetime'] < train_end)].copy()
    test_df = df[(df['Datetime'] >= train_end) & (df['Datetime'] <= test_end)].copy()

    if len(train_df) < 30 or len(test_df) < 30:
        return None, None, None, None, None

    features = df.select_dtypes(include=[np.number]).columns.tolist()
    features = [col for col in features if col not in ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']]

    X_train = train_df[features]
    X_test = test_df[features]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(X_train_scaled)
    test_df['cluster'] = kmeans.predict(X_test_scaled)

    test_df['return'] = test_df['Close'].pct_change().fillna(0)
    test_df['strategy'] = test_df['cluster'].shift(1).fillna(0) * test_df['return']
    test_df['cumulative_market'] = (1 + test_df['return']).cumprod()
    test_df['cumulative_strategy'] = (1 + test_df['strategy']).cumprod()
    test_df['Forward_Return'] = test_df['Close'].pct_change(periods=10).shift(-10)

    cluster_returns = test_df.groupby('cluster')['Forward_Return'].mean().sort_values(ascending=False)
    signal_labels = ['Buy', 'Hold', 'Sell']
    signal_map = {cluster: signal_labels[i] if i < len(signal_labels) else 'Hold'
                  for i, cluster in enumerate(cluster_returns.index)}
    test_df['signal'] = test_df['cluster'].map(signal_map)
    test_df['Portfolio_Value'] = 100000 * (1 + test_df['strategy']).cumprod()

    # Evaluation Metrics
    test_df['True_Label'] = (test_df['Forward_Return'] > 0).astype(int)
    signal_to_pred = {'Buy': 1, 'Hold': 0, 'Sell': 0}
    test_df['Predicted_Label'] = test_df['signal'].map(signal_to_pred).fillna(0).astype(int)

    accuracy = accuracy_score(test_df['True_Label'], test_df['Predicted_Label'])
    precision = precision_score(test_df['True_Label'], test_df['Predicted_Label'], zero_division=0)
    recall = recall_score(test_df['True_Label'], test_df['Predicted_Label'], zero_division=0)
    f1 = f1_score(test_df['True_Label'], test_df['Predicted_Label'], zero_division=0)

    stats = {
        "Final Market": test_df['cumulative_market'].iloc[-1],
        "Final Strategy": test_df['cumulative_strategy'].iloc[-1],
        "Sharpe": test_df['strategy'].mean() / (test_df['strategy'].std() + 1e-6) * np.sqrt(252),
        "Drawdown": (test_df['cumulative_strategy'].cummax() - test_df['cumulative_strategy']).max(),
        "Final Portfolio Value": test_df['Portfolio_Value'].iloc[-1],
        "Signal Map": signal_map,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1
    }

    return stats, test_df, kmeans, scaler, list(X_train.columns)

# === Main Loop ===
summary = []

for ticker in TICKERS:
    print(f"\n Processing {ticker}")
    df = yf.download(ticker, period="730d", interval="1h", progress=False)
    if df is None or df.empty:
        print(f" Skipping {ticker}, no data.")
        continue

    df.reset_index(inplace=True)
    df['Datetime'] = pd.to_datetime(df['Datetime'])

    stats, result_df, model, scaler, feature_names = walkforward_kmeans(df)
    if result_df is None:
        print(f" Skipping {ticker}, insufficient rows.")
        continue

    stats.update({
        "Ticker": ticker,
        "Model": "KMeans",
        "Return": result_df['strategy'].sum()
    })
    print(f"[DEBUG] Saving {ticker} - Final Portfolio: {stats['Final Portfolio Value']:.2f}")
    summary.append(stats)

    result_df.to_csv(f"{RESULTS_DIR}/data/{ticker}_result.csv", index=False)
    joblib.dump(model, f"{RESULTS_DIR}/models/{ticker}_model.pkl")
    joblib.dump(scaler, f"{RESULTS_DIR}/models/{ticker}_scaler.pkl")
    with open(f"{RESULTS_DIR}/models/{ticker}_features.txt", 'w') as f:
        json.dump(list(map(str, feature_names)), f)

    joblib.dump(model, os.path.join(FINAL_MODEL_DIR, f"kmeans_{ticker}.pkl"))
    joblib.dump(scaler, os.path.join(FINAL_MODEL_DIR, f"kmeans_{ticker}_scaler.pkl"))
    with open(os.path.join(FINAL_MODEL_DIR, f"kmeans_{ticker}_features.json"), "w") as f:
        json.dump(list(map(str, feature_names)), f)

    # Save plot per ticker
    plt.figure(figsize=(12, 6))
    plt.plot(result_df['cumulative_market'], label='Market (Buy & Hold)', linestyle='-')
    plt.plot(result_df['cumulative_strategy'], label='KMeans Strategy', linestyle='--')
    plt.title(f"{ticker} - Cumulative Returns")
    plt.xlabel("Time Steps")
    plt.ylabel("Cumulative Return")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plot_path = f"{RESULTS_DIR}/plots/{ticker}_portfolio_plot.png"
    plt.savefig(plot_path, dpi=300)
    plt.close()
    print(f" Plot saved: {plot_path}")

    gc.collect()

# === Summary and Ranking ===
summary_df = pd.DataFrame(summary)
summary_df['score'] = (
    summary_df['Sharpe'] * 0.4 +
    summary_df['Final Strategy'] * 0.3 +
    summary_df['Final Portfolio Value'] * 0.3
)
summary_df = summary_df.sort_values('score', ascending=False)
summary_df.to_csv(f"{RESULTS_DIR}/summary_forward_return_remap.csv", index=False)
summary_df.to_csv(f"{RESULTS_DIR}/model_selector_metrics.csv", index=False)

best_models = summary_df.sort_values(['Ticker', 'score'], ascending=[True, False])\
                        .groupby('Ticker').first().reset_index()
best_models.to_excel(f"{RESULTS_DIR}/best_models_by_score.xlsx", index=False)



 Processing AAPL
[DEBUG] Saving AAPL - Final Portfolio: 92479.29
 Plot saved: /content/drive/MyDrive/Results_May_2025/kmeans_walkforward_results/plots/AAPL_portfolio_plot.png

 Processing TSLA
[DEBUG] Saving TSLA - Final Portfolio: 54832.74
 Plot saved: /content/drive/MyDrive/Results_May_2025/kmeans_walkforward_results/plots/TSLA_portfolio_plot.png

 Processing MSFT
[DEBUG] Saving MSFT - Final Portfolio: 96829.32
 Plot saved: /content/drive/MyDrive/Results_May_2025/kmeans_walkforward_results/plots/MSFT_portfolio_plot.png

 Processing GOOGL
[DEBUG] Saving GOOGL - Final Portfolio: 81061.66
 Plot saved: /content/drive/MyDrive/Results_May_2025/kmeans_walkforward_results/plots/GOOGL_portfolio_plot.png

 Processing AMZN
[DEBUG] Saving AMZN - Final Portfolio: 113742.48
 Plot saved: /content/drive/MyDrive/Results_May_2025/kmeans_walkforward_results/plots/AMZN_portfolio_plot.png

 Processing NVDA
[DEBUG] Saving NVDA - Final Portfolio: 106125.00
 Plot saved: /content/drive/MyDrive/Results_May_2